# 28 · Dynamic INT8 and CPU batching

Runs [`docs/experiments/int8_and_batching/PROTOCOL.md`](../docs/experiments/int8_and_batching/PROTOCOL.md),
locked before any run, in about **20–30 minutes**. Use a **T4 GPU runtime** (Runtime → Change
runtime type → T4 GPU): nothing runs on the GPU, but its CPU is the one the project's other
timings used.

It checks out the pinned code commit, rebuilds the random split and stops unless it is the one
the model was scored on, downloads the published DistilBERT at its pinned revision and checks its
weights hash. Then `scripts/benchmark_int8.py` measures FP32 against dynamic INT8 on all 13,263
test reviews and times both on one CPU thread.

**Send back** `int8_and_batching_results.zip`. It holds logits, timings, sizes and the
comparison, not model weights.

In [ ]:
from pathlib import Path

SOURCE_COMMIT = "93dfdff49545faf81a71386baaeaf8a45d889d21"
REPO = Path("/content/hotel-review-nlp")
MODEL_DIR = Path("/content/models/distilbert")  # outside the repository: weights are never bundled

In [ ]:
import os
import subprocess
import sys

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/krimits/hotel-review-nlp.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if GIT_COMMIT != SOURCE_COMMIT:
    raise RuntimeError(f"checked out {GIT_COMMIT}, not the pinned {SOURCE_COMMIT}")
sys.path.insert(0, str(REPO / "src"))
# The transformers version of the phase 2 runs and of the CI.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.56.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
print("commit", GIT_COMMIT)

## 1. Data: the random split the model was scored on

In [ ]:
import json

from reviewnlp.utils.experiments import fingerprint_splits
from reviewnlp.utils.seed import load_config


def run(command: str) -> None:
    print("$", command, flush=True)
    subprocess.run(command, shell=True, check=True)


DATA_CONFIG = "configs/baselines.yaml"
views = json.loads(Path("docs/experiments/results/split_views.json").read_text(encoding="utf-8"))
processed = load_config(DATA_CONFIG)["data"]["processed_dir"]
run("python scripts/fetch_booking_515k.py")
if not Path(processed, "test.parquet").exists():
    run(f"python -m reviewnlp.data.preprocess --config {DATA_CONFIG}")
if fingerprint_splits(processed) != views["random"]["splits"]:
    raise RuntimeError("The random split differs from the one the model was scored on. Stopping.")
print("random: the same reviews, in the same order, as in split_views.json")

## 2. The published model, at its pinned revision

In [ ]:
import hashlib

from huggingface_hub import snapshot_download

sys.path.insert(0, str(REPO / "scripts"))
import benchmark_int8 as int8

snapshot_download(int8.MODEL_REPO, revision=int8.MODEL_REVISION, local_dir=MODEL_DIR,
                  allow_patterns=["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json",
                                  "special_tokens_map.json", "vocab.txt"])
if int8.weights_sha256(MODEL_DIR) != int8.WEIGHTS_SHA256:
    raise RuntimeError("The downloaded weights are not the published checkpoint. Stopping.")
print("weights sha256 matches", int8.WEIGHTS_SHA256[:12])

## 3. The measurement (about 15–25 minutes)

The whole test set is labelled twice, by FP32 and by INT8, then both are timed on one thread on
500 reviews. The run stops unless FP32 matches the committed test logits on at least 99%.

In [ ]:
OUT = Path("docs/experiments/results/int8_v1")
run(f"python scripts/benchmark_int8.py --model {MODEL_DIR} --processed-dir {processed} --out-dir {OUT}")

## 4. Collect and download

In [ ]:
import platform
import zipfile
from importlib.metadata import version

import torch

environment = {
    "commit": GIT_COMMIT, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "python": platform.python_version(),
    "packages": {name: version(name) for name in ("torch", "transformers", "tokenizers", "numpy", "pandas")},
    "model": {"repo": int8.MODEL_REPO, "revision": int8.MODEL_REVISION, "weights_sha256": int8.WEIGHTS_SHA256}}
(OUT / "environment.json").write_text(json.dumps(environment, indent=1) + "\n", encoding="utf-8")

collected = sorted(path for path in OUT.iterdir() if path.is_file() and path.name != "artifact_manifest.json")
if any(path.suffix in (".safetensors", ".bin", ".pt") for path in collected):
    raise RuntimeError("model weights in the bundle")
manifest = {"commit": GIT_COMMIT,
            "files": [{"path": path.as_posix(), "bytes": path.stat().st_size,
                       "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in collected]}
(OUT / "artifact_manifest.json").write_text(json.dumps(manifest, indent=1) + "\n", encoding="utf-8")
with zipfile.ZipFile("int8_and_batching_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [*collected, OUT / "artifact_manifest.json"]:
        archive.write(path, path.as_posix())
print(f"{len(collected) + 1} files")

In [ ]:
from google.colab import files

files.download("int8_and_batching_results.zip")

## 5. After the run

Send `int8_and_batching_results.zip` back. Its paths are the repository's: the bundle is checked
against `artifact_manifest.json` and the commit, the comparison is recomputed from the logits,
and the protocol's rule is recorded exactly as it fired, in a decision note.